# Tabular Data Generation Tutorial with Ligen

### Course: 114-1 Mobile and Pervasive Intelligence
### Instructor: Dr. Fang-Jing Wu
### Teaching Assistant: Yoyo Lee
---

This tutorial shows how to use **Ligen** — a GAN-based toolkit for tabular data generation — to augment real-world datasets and improve downstream model performance.
We'll walk through: creating an environment, preparing data, running a short training example (via `main.py`).

Supported models in this repo include FreeGAN, PointGAN and a simple MLP baseline for experiments.

Files you'll interact with:
- `main.py` — command-line entrypoint; accepts a YAML config from `experiment/`.
- `Data/` — sample CSVs: `datasample_spectral.csv`, `datasample_wifi.csv`.
- `requirements.txt` — Python dependencies.
---



## Step 1: Create and install the Python environment

Recommended: create an isolated environment (venv or conda) to avoid dependency conflicts.

Examples (pick one):

Using venv (macOS / zsh):

```bash
python3 -m venv .venv
source .venv/bin/activate
```

Using conda:

```bash
conda create -n ligen python=3.10 -y
conda activate ligen
```

Then install dependencies:

```bash
pip install -r requirements.txt
```

If you don't want to create an isolated env, just run the following block.

In [1]:
# Or if you don't want to use a virtual environment, just run:
import sys
import subprocess
print('Using Python executable:', sys.executable)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
print('Finished installing dependencies')

Using Python executable: /Users/yl/Ligen/.venv/bin/python
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Finished installing dependencies


## Step 2: Inspect the Sample Data

The `Data/` folder contains small example CSV files for testing. Let's explore their structure:

**Note**: These are minimal sample datasets (with repeated rows) for demonstration purposes only. For real training, you'll need larger, more diverse datasets with hundreds or thousands of unique samples.

In [2]:
import pandas as pd
pd.read_csv('Data/datasample_wifi.csv').head()

,positionX,positionY,sample_id,7C:10:C9:B7:59:5C,EC:58:EA:19:10:B8,EC:58:EA:19:10:BC,EC:58:EA:59:10:BC,EC:58:EA:99:00:78,EC:58:EA:99:10:BC
0,0.0,0.0,0,-80.0,-52.0,-52.0,-52.0,-56.0,-52.0
1,1.0,0.0,1,-82.0,-54.0,-54.0,-54.0,-58.0,-54.0
2,2.0,0.0,2,-84.0,-56.0,-56.0,-56.0,-60.0,-56.0
3,3.0,0.0,3,-86.0,-58.0,-58.0,-58.0,-62.0,-58.0
4,4.0,0.0,4,-88.0,-60.0,-60.0,-60.0,-64.0,-60.0


In [3]:
pd.read_csv('Data/datasample_spectral.csv').head()

,F1,F2,F3,F4,F5,F6,F7,F8,Clear,NIR,x,y
0,16,26,41,62,167,136,145,56,402,77,0.0,0.0
1,16,26,41,62,167,136,145,56,402,76,0.0,0.0
2,16,26,41,63,167,136,145,56,402,77,0.0,0.0
3,16,26,41,62,167,136,145,56,402,77,0.0,0.0
4,16,26,41,62,168,136,145,56,402,77,0.0,0.0


## Step 3: Quick Start - Command Line Usage

You can run Ligen training directly from the terminal using the provided configuration files:

```bash
# Train with WiFi data
python main.py --config experiment/wifi.yaml

# Train with spectral data  
python main.py --config experiment/spectral.yaml
```

The training process will:
1. Load and preprocess your data
2. Train the selected GAN models
3. Generate synthetic samples
4. Save results and model checkpoints

**Training time**: Depends on your hardware and dataset size. The sample data trains quickly (~1-2 minutes), but real datasets may take hours.

Let's examine the configuration files to understand how to customize the training process:

## Step 4: Understanding Configuration Files

Ligen uses YAML configuration files to define experiment settings. There are two example configs in the `experiment/` directory:
- `experiment/spectral.yaml` - for spectral sensor data
- `experiment/wifi.yaml` - for WiFi RSSI positioning data

### Key Configuration Sections

Each config file contains several important sections:

**1. Dataset Configuration:**
- `kind`: Specifies the dataset type (`spectral` or `wifi`)
- `path`: Path to your CSV data file
- `x_cols`: Feature columns (input variables)
- `y_cols`: Target columns (coordinates for pointgan conditioning)

Note that WiFi and spectral datasets have different column names, so you'll need to adjust the `x_cols` and `y_cols` accordingly when working with your own data.

In [10]:
# Let's examine the configuration files to understand their structure
import yaml

# Read and display the spectral config
print("=== SPECTRAL CONFIG (experiment/spectral.yaml) ===")
with open("experiment/spectral.yaml", 'r') as f:
    spectral_config = yaml.safe_load(f)
    
# Pretty print the config structure
import json
print(json.dumps(spectral_config['dataset'], indent=2))

print("\n" + "="*60 + "\n")

# Read and display the wifi config  
print("=== WIFI CONFIG (experiment/wifi.yaml) ===")
with open("experiment/wifi.yaml", 'r') as f:
    wifi_config = yaml.safe_load(f)
    
print(json.dumps(wifi_config['dataset'], indent=2))

=== SPECTRAL CONFIG (experiment/spectral.yaml) ===
{
  "kind": "spectral",
  "path": "Data/datasample_spectral.csv",
  "x_cols": [
    "F1",
    "F2",
    "F3",
    "F4",
    "F5",
    "F6",
    "F7",
    "F8",
    "Clear",
    "NIR"
  ],
  "y_cols": [
    "x",
    "y"
  ]
}


=== WIFI CONFIG (experiment/wifi.yaml) ===
{
  "kind": "wifi",
  "path": "Data/datasample_wifi.csv",
  "x_cols": [
    "7C:10:C9:B7:59:5C",
    "EC:58:EA:19:10:B8",
    "EC:58:EA:19:10:BC",
    "EC:58:EA:59:10:BC",
    "EC:58:EA:99:00:78",
    "EC:58:EA:99:10:BC"
  ],
  "y_cols": [
    "positionX",
    "positionY"
  ]
}


**2. Global Experiment Settings:**
- `base_dir`: Directory where results will be saved
- `name`: Unique experiment name for organizing outputs
- `wandb`: A tool for online visualize training progress. (recommend to use this if you have an account, but it might take a while.)
- `device`: You can accelerate the training proces using gpu, but setting `auto` would have your device find the best cpu/gpu to run with.
- `seeds`: How many experiments you want to run for each portion.

In [ ]:
# Let's examine the configuration files to understand their structure
import yaml

# Read and display the spectral config
with open("experiment/spectral.yaml", 'r') as f:
    spectral_config = yaml.safe_load(f)
    
# Pretty print the config structure
import json
# drop dataset
spectral_config.pop('dataset', None)
spectral_config.pop('freegan', None)
spectral_config.pop('pointgan', None)
spectral_config.pop('mlp', None)
print(json.dumps(spectral_config, indent=2))

=== SPECTRAL CONFIG (experiment/spectral.yaml) ===
{
  "experiment": {
    "base_dir": "experiment",
    "name": "Spectral_20241126_clean"
  },
  "wandb": {
    "enabled": false,
    "project": "localizer"
  },
  "device": "auto",
  "batch_size": 2,
  "seeds": 2
}


**3. Model Specific Settings**

⚠️ **IMPORTANT**: The default configuration files only contain parameters for demonstrations. For the sample datasets provided, you **must** modify these parameters to avoid training issues and long wait times.

### Model-Specific Parameters:

**FreeGAN** (Unconditional tabular data generation):
- `epochs`: Training epochs
- `lr`: Learning rate 
- `z_dim`: Latent noise dimension
- `hidden_size`: Network width
- `n_synth`: Number of synthetic samples to generate
- `d_steps`: Discriminator training steps per generator step
- `label_smoothing`: Smoothing factor for real labels
- `gen_use_bn`: Whether to use Batch Normalization in generator (false for small datasets)

**PointGAN** (Coordinate-conditioned generation):
- `epochs`: Training epochs 
- `lr`: Learning rate
- `noise_dim`: Noise vector dimension 
- `coord_dim`: Coordinate space dimension (usually 2 for x,y)
- `fid_mse_weight`: Balance between quality and coordinate accuracy
- `synth_multi`: Multiplier for synthetic data generation

**MLP** (Baseline model with Optuna hyperparameter optimization):

The MLP uses **Optuna** for automatic hyperparameter tuning. Optuna tries different combinations of parameters to find the best configuration:

*Core Parameters:*
- `hidden_min/max`: Hidden layer size search range (e.g., 16-1024)
- `lr_min/max`: Learning rate search range (e.g., 0.0001-0.001)
- `dropout_min/max`: Dropout rate search range (e.g., 0.0-0.5)

*Optuna-Specific Settings:*
- `optuna_trials`: Number of different hyperparameter combinations to try (default: 5)
- `optuna_epochs`: Training epochs per trial (not the same as `epoch_cnt`)
- `epoch_cnt`: Total training epochs for final model (after finding best params)

*How Optuna Works in MLP:*
1. **Search Phase**: Tries `optuna_trials` different combinations of hidden_size, learning_rate, and dropout
2. **Each trial**: Trains for `optuna_epochs` to evaluate performance
3. **Selection**: Picks the best hyperparameter combination
4. **Final Training**: Trains the final model with best params for `epoch_cnt` epochs

*Sample vs Production Settings:*
```yaml
# For sample data (fast testing)
optuna_trials: 3        # Try fewer combinations
optuna_epochs: 50       # Short evaluation per trial
epoch_cnt: 200          # Final training epochs

# For production data (thorough search)
optuna_trials: 50       # Try many combinations
optuna_epochs: 200      # Longer evaluation per trial  
epoch_cnt: 2000         # Extensive final training
```

### Quick Fix for Sample Data:
The provided configurations are set for production datasets. For the tutorial samples, create a copy of the config file and modify these key values to reasonable ranges for small datasets.

In [17]:
# Demonstrate Optuna parameter ranges and what each trial tests
import yaml
import json

print("🔍 MLP + OPTUNA PARAMETER BREAKDOWN")
print("="*50)

# Load configs to show actual parameter ranges
with open("experiment/spectral.yaml", 'r') as f:
    spectral_config = yaml.safe_load(f)

with open("experiment/wifi.yaml", 'r') as f:
    wifi_config = yaml.safe_load(f)

print("\n📊 Spectral Dataset MLP Config:")
spectral_mlp = spectral_config.get('mlp', {})
print(json.dumps(spectral_mlp, indent=2))

print("\n📊 WiFi Dataset MLP Config:")
wifi_mlp = wifi_config.get('mlp', {})
print(json.dumps(wifi_mlp, indent=2))

print("\n🎯 OPTUNA SEARCH PROCESS:")
print("="*30)

def simulate_optuna_trial(trial_num, hidden_min, hidden_max, lr_min, lr_max, dropout_min, dropout_max):
    """Simulate what Optuna might select in a trial"""
    import random
    import math
    
    # Simulate log-uniform sampling for hidden size and learning rate
    hidden = int(math.exp(random.uniform(math.log(hidden_min), math.log(hidden_max))))
    lr = math.exp(random.uniform(math.log(lr_min), math.log(lr_max)))
    dropout = random.uniform(dropout_min, dropout_max)
    
    return hidden, lr, dropout

print("Example of what Optuna trials might test (WiFi config):")
print("Trial | Hidden Size | Learning Rate | Dropout | Process")
print("-" * 65)

for i in range(5):
    hidden, lr, dropout = simulate_optuna_trial(
        i, wifi_mlp.get('hidden_min', 128), wifi_mlp.get('hidden_max', 1024),
        wifi_mlp.get('lr_min', 0.0001), wifi_mlp.get('lr_max', 0.001),
        wifi_mlp.get('dropout_min', 0.0), wifi_mlp.get('dropout_max', 0.5)
    )
    
    epoch_cnt = wifi_mlp.get('optuna_trials', 5) * 200  # Simulate optuna_epochs
    print(f"  {i+1}   |    {hidden:4d}     |   {lr:.6f}    |  {dropout:.3f}  | Train {epoch_cnt}→eval")

print(f"\n🏆 After {wifi_mlp.get('optuna_trials', 5)} trials, Optuna picks BEST params")
print(f"📈 Then trains final model for {wifi_mlp.get('epoch_cnt', 2000)} epochs")

print("\n💡 KEY INSIGHTS:")
print("- Each trial tests a DIFFERENT network architecture")
print("- Optuna uses Bayesian optimization to guide the search")
print("- Early trials are more random, later trials focus on promising regions")
print("- Sample data should use fewer trials (3-5) for speed")
print("- Production data can use many trials (20-100) for best performance")

🔍 MLP + OPTUNA PARAMETER BREAKDOWN

📊 Spectral Dataset MLP Config:
{
  "hidden_min": 16,
  "hidden_max": 64,
  "lr_min": 1e-05,
  "lr_max": 0.001,
  "dropout_min": 0.1,
  "dropout_max": 0.3,
  "epoch_cnt": 30,
  "optuna_trials": 5,
  "optuna_epochs": 50
}

📊 WiFi Dataset MLP Config:
{
  "hidden_min": 128,
  "hidden_max": 1024,
  "lr_min": 0.0001,
  "lr_max": 0.001,
  "dropout_min": 0.0,
  "dropout_max": 0.5,
  "epoch_cnt": 2000,
  "optuna_trials": 5,
  "optuna_epochs": 100
}

🎯 OPTUNA SEARCH PROCESS:
Example of what Optuna trials might test (WiFi config):
Trial | Hidden Size | Learning Rate | Dropout | Process
-----------------------------------------------------------------
  1   |     740     |   0.000573    |  0.210  | Train 1000→eval
  2   |     219     |   0.000325    |  0.202  | Train 1000→eval
  3   |     653     |   0.000201    |  0.238  | Train 1000→eval
  4   |     430     |   0.000809    |  0.252  | Train 1000→eval
  5   |     230     |   0.000570    |  0.309  | Train 1000→e

In [18]:
# Let's examine the configuration files to understand their structure
import yaml

# Read and display the spectral config
with open("experiment/spectral.yaml", 'r') as f:
    spectral_config = yaml.safe_load(f)
    
# Pretty print the config structure
import json
# drop dataset
spectral_config.pop('dataset', None)
spectral_config.pop('experiment', None)
spectral_config.pop('wandb', None)
spectral_config.pop('device', None)
spectral_config.pop('seeds', None)
spectral_config.pop('device', None)
spectral_config.pop('batch_size', None)
print(json.dumps(spectral_config, indent=2))

{
  "freegan": {
    "z_dim": 10,
    "hidden_size": 256,
    "lr": 0.0002,
    "betas": [
      0.5,
      0.999
    ],
    "label_smoothing": 0.9,
    "synth_multi": 2,
    "n_synth": 200,
    "d_steps": 2,
    "epochs": 10,
    "save_every": 1000,
    "gen_use_bn": false
  },
  "pointgan": {
    "noise_dim": 10,
    "coord_dim": 2,
    "lr": 1e-05,
    "d_steps": 2,
    "fid_mse_weight": 0.1,
    "synth_multi": 2,
    "epochs": 10
  },
  "mlp": {
    "hidden_min": 16,
    "hidden_max": 64,
    "lr_min": 1e-05,
    "lr_max": 0.001,
    "dropout_min": 0.1,
    "dropout_max": 0.3,
    "epoch_cnt": 30,
    "optuna_trials": 5,
    "optuna_epochs": 50
  }
}


In [7]:
# Run a quick training example with the spectral dataset
# This will train both pointgan and freegan models

import sys
import os

# Add the current directory to Python path
sys.path.append('.')

try:
    print("Starting model training...")
    print("This may take a few minutes depending on your hardware.\n")
    
    # Choose which config to use
    config_path = "experiment/spectral.yaml"
    print(f"Using configuration: {config_path}")
    
    # Import and run the models
    from runners.pointgan_runner import run as run_pointgan
    from runners.freegan_runner import run as run_freegan
    
    print("\n1. Training pointgan model...")
    run_pointgan(config_path)
    
    print("\n2. Training freegan model...")  
    run_freegan(config_path)
    
    print("\n✅ Training completed! Check the Results/ directory for outputs.")
    
except Exception as e:
    print(f"❌ Error during training: {str(e)}")
    print("This is normal if running with small sample data.")
    print("For real training, use larger datasets with more diverse samples.")

Starting model training...
This may take a few minutes depending on your hardware.

Using configuration: experiment/spectral.yaml

1. Training pointgan model...
01:11:14 | INFO | ligen: Logger initialized. File: experiment/pointgan-Spectral_20241126_clean-20251008_011114/logs/pointgan-Spectral_20241126_clean-20251008_011114_20251008-011114.log
01:11:14 | INFO | ligen.pointgan: Active log files: ['/Users/yl/Ligen/experiment/pointgan-Spectral_20241126_clean-20251008_011114/logs/pointgan-Spectral_20241126_clean-20251008_011114_20251008-011114.log']
01:11:14 | INFO | ligen.pointgan: Copied config to experiment/pointgan-Spectral_20241126_clean-20251008_011114/spectral.yaml and experiment/pointgan-Spectral_20241126_clean-20251008_011114/run_config.yaml
01:11:14 | INFO | ligen.pointgan: Loaded config from experiment/spectral.yaml
01:11:14 | INFO | ligen.pointgan: Saving results to Results/pointgan-spectral-pointgan-Spectral_20241126_clean-20251008_011114-results.csv, experiment dirs: experime

[pointgan] Training:  20%|██        | 2/10 [00:03<00:12,  1.56s/it, D=0.0090 G=4001.1147]

KeyboardInterrupt: 

## Step 5: Understanding the Results

After training, Ligen saves several outputs:

### Generated Files:
- **Model checkpoints**: Saved in `experiment/<experiment_name>/model/`
- **Generated data**: Synthetic samples in `experiment/<experiment_name>/generated_data/`
- **Results CSV**: Performance metrics in `Results/` directory

### Key Metrics:
- **MSE (Mean Squared Error)**: For coordinate prediction accuracy
- **Training loss**: Convergence indicator

### Next Steps:
1. **Use your own data**: Replace the sample CSVs with your datasets
2. **Adjust hyperparameters**: Modify the YAML configs for better performance
3. **Scale up training**: Increase epochs and batch_size for production use
4. **Evaluate results**: Compare synthetic vs. real data distributions

---

## Step 6: Running from Command Line

You can also run Ligen directly from the terminal:

In [6]:
# You can run these commands in a terminal:
# python main.py --config experiment/wifi.yaml
# python main.py --config experiment/spectral.yaml

# Or run directly with custom arguments:
import subprocess
import sys

def run_ligen_command(config_file):
    """Run Ligen training from Python"""
    cmd = [sys.executable, "main.py", "--config", config_file]
    print(f"Running: {' '.join(cmd)}")
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        print("STDOUT:", result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr)
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print("⚠️  Training timeout (5 min limit for demo)")
        return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# Uncomment to run:
# success = run_ligen_command("experiment/spectral.yaml")
# print(f"Training {'succeeded' if success else 'failed'}")

## Step 7: Advanced Usage & Tips

### Working with Your Own Data

1. **Prepare your CSV file** with appropriate columns
2. **Update the config file**:
   ```yaml
   dataset:
     kind: custom  # or wifi/spectral
     path: path/to/your/data.csv
     x_cols: [feature1, feature2, feature3]  # your feature columns
     y_cols: [target1, target2]              # your target columns
   ```

3. **Adjust hyperparameters** based on your dataset size and complexity

### Common Issues & Solutions

- **Out of memory**: Reduce `batch_size` in config
- **Poor quality**: Increase `epochs` and adjust learning rates
- **Slow training**: Use GPU by setting `device: cuda`
- **Import errors**: Ensure all dependencies are installed

### Model Comparison

- **FreeGAN**: High variance, but sometimes good performance.
- **PoinGAN**: Low variance, more stable.
- **MLP**: Baseline for performance comparison

Happy generating! 🚀

## Appendix: Single Data Inference

This section demonstrates how to load trained models and make predictions on individual data points. You can use any of the three model types:

### 🧠 **MLP Inference**
- **Input**: Feature values (sensor readings)
- **Output**: Predicted coordinates (x, y)
- **Use case**: Predict location from sensor measurements

### 🎲 **PointGAN Inference** 
- **Input**: Target coordinates + random noise
- **Output**: Generated feature values
- **Use case**: Generate synthetic sensor readings for a specific location

### 🆓 **FreeGAN Inference**
- **Input**: Random noise vector
- **Output**: Generated feature values  
- **Use case**: Generate completely synthetic sensor readings

### 📝 **How to Use:**

1. **Set MODEL_TYPE**: Choose `"mlp"`, `"pointgan"`, or `"freegan"`
2. **Update MODEL_PATHS**: Point to your trained model files
3. **Modify INPUT_FEATURES**: Set your sensor values or coordinates
4. **Run the cells**: Execute the inference code

⚠️ **Important**: Make sure you have trained models first using the previous steps!

In [23]:
# Single Data Inference - Load Trained Models and Make Predictions
import torch
import pandas as pd
import numpy as np
import os
import sys

# Add the current directory to Python path for imports
sys.path.append('.')

def load_mlp_model(model_path, device='cpu'):
    """Load a trained MLP model"""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    return model

def load_pointgan_generator(checkpoint_path, coord_dim=2, feat_dim=10, noise_dim=10, device='cpu'):
    """Load a trained PointGAN generator"""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint file not found: {checkpoint_path}")
    
    # Import the generator model
    from models.pointgan import Generator
    
    # Create generator instance
    generator = Generator(coord_dim=coord_dim, feat_dim=feat_dim, noise_dim=noise_dim)
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    generator.load_state_dict(checkpoint['G'])
    generator.to(device)
    generator.eval()
    
    return generator

def load_freegan_generator(checkpoint_path, z_dim=10, output_size=10, hidden_size=256, device='cpu'):
    """Load a trained FreeGAN generator"""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint file not found: {checkpoint_path}")
    
    # Import the generator model  
    from models.freegan import Generator
    
    # Create generator instance (you may need to adjust cfg parameters)
    class DummyCfg:
        def __init__(self):
            pass
    cfg = DummyCfg()
    
    generator = Generator(z_dim=z_dim, hidden_size=hidden_size, output_size=output_size, cfg=cfg)
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    generator.load_state_dict(checkpoint['G'])
    generator.to(device)
    generator.eval()
    
    return generator

# =============================================================================
# MANUAL CONFIGURATION SECTION - EDIT THESE VALUES
# =============================================================================

# 1. Choose your model type and paths
MODEL_TYPE = "mlp"  # Options: "mlp", "pointgan", "freegan"

# 2. Model paths (update these to your actual trained model paths)
MODEL_PATHS = {
    "mlp": "experiment/pointgan-Spectral_20241126_clean-20251008_010709/model/MLP_pipeline_1_p9_s0",
    "pointgan": "experiment/pointgan-Spectral_20241126_clean-20251008_010709/model/pointgan_p9_s0.pth", 
    "freegan": "experiment/freegan-Spectral_20241126_clean-20251008_004059/model/freegan_p9_s0.pth"
}

# 3. Input features for prediction (spectral sensor data example)
# Update these values with your own sensor readings
INPUT_FEATURES = {
    "F1": 0.5,
    "F2": 0.3, 
    "F3": 0.8,
    "F4": 0.2,
    "F5": 0.9,
    "F6": 0.1,
    "F7": 0.6,
    "F8": 0.4,
    "Clear": 0.7,
    "NIR": 0.35
}

# 4. Device configuration
DEVICE = "cpu"  # Change to "cuda" if you have GPU

print(f"🔮 SINGLE DATA INFERENCE")
print(f"Model Type: {MODEL_TYPE}")
print(f"Device: {DEVICE}")
print(f"Input Features: {INPUT_FEATURES}")
print("=" * 60)

🔮 SINGLE DATA INFERENCE
Model Type: mlp
Device: cpu
Input Features: {'F1': 0.5, 'F2': 0.3, 'F3': 0.8, 'F4': 0.2, 'F5': 0.9, 'F6': 0.1, 'F7': 0.6, 'F8': 0.4, 'Clear': 0.7, 'NIR': 0.35}


In [24]:
# Execute the inference based on model type
try:
    model_path = MODEL_PATHS[MODEL_TYPE]
    print(f"📂 Loading model from: {model_path}")
    
    if MODEL_TYPE == "mlp":
        # ===== MLP INFERENCE =====
        print("\n🧠 MLP Model Inference")
        
        # Load the trained MLP model
        model = load_mlp_model(model_path, device=DEVICE)
        print(f"✅ Model loaded successfully")
        print(f"   - Input size: {model.fc1.in_features}")
        print(f"   - Hidden size: {model.fc1.out_features}")
        print(f"   - Output size: {model.fc5.out_features}")
        
        # Prepare input data
        input_values = list(INPUT_FEATURES.values())
        input_tensor = torch.tensor(input_values, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        
        print(f"\n📊 Input shape: {input_tensor.shape}")
        print(f"📊 Input values: {input_values}")
        
        # Make prediction
        with torch.no_grad():
            prediction = model(input_tensor)
            pred_coords = prediction.cpu().numpy().flatten()
        
        print(f"\n🎯 PREDICTION RESULTS:")
        print(f"   Predicted X coordinate: {pred_coords[0]:.4f}")
        print(f"   Predicted Y coordinate: {pred_coords[1]:.4f}")
        print(f"   Predicted location: ({pred_coords[0]:.4f}, {pred_coords[1]:.4f})")
        
    elif MODEL_TYPE == "pointgan":
        # ===== POINTGAN INFERENCE =====
        print("\n🎲 PointGAN Model Inference")
        
        # You need to specify these parameters based on your training config
        COORD_DIM = 2  # Usually 2 for (x, y) coordinates
        FEAT_DIM = len(INPUT_FEATURES)  # Number of input features
        NOISE_DIM = 10  # From your config file
        
        # Load the trained PointGAN generator
        generator = load_pointgan_generator(
            model_path, 
            coord_dim=COORD_DIM, 
            feat_dim=FEAT_DIM, 
            noise_dim=NOISE_DIM, 
            device=DEVICE
        )
        print(f"✅ PointGAN Generator loaded successfully")
        
        # For PointGAN, you need to provide both coordinates and noise to generate features
        # Here we'll demonstrate generating features given a coordinate
        target_coords = [0.5, 0.5]  # Example target coordinates
        print(f"🎯 Target coordinates: {target_coords}")
        
        # Prepare inputs
        coords_tensor = torch.tensor(target_coords, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        noise_tensor = torch.randn(1, NOISE_DIM).to(DEVICE)
        
        # Generate features
        with torch.no_grad():
            generated_features = generator(coords_tensor, noise_tensor)
            gen_features = generated_features.cpu().numpy().flatten()
        
        print(f"\n🎯 GENERATION RESULTS:")
        print(f"   Input coordinates: ({target_coords[0]}, {target_coords[1]})")
        print(f"   Generated features: {gen_features}")
        
        # Create a DataFrame for better visualization
        feature_names = list(INPUT_FEATURES.keys())
        result_df = pd.DataFrame([gen_features], columns=feature_names)
        print(f"\n📊 Generated Feature Values:")
        print(result_df.round(4))
        
    elif MODEL_TYPE == "freegan":
        # ===== FREEGAN INFERENCE =====
        print("\n🆓 FreeGAN Model Inference")
        
        # You need to specify these parameters based on your training config
        Z_DIM = 10  # From your config file
        OUTPUT_SIZE = len(INPUT_FEATURES)  # Number of features to generate
        HIDDEN_SIZE = 256  # From your config file
        
        # Load the trained FreeGAN generator
        generator = load_freegan_generator(
            model_path,
            z_dim=Z_DIM,
            output_size=OUTPUT_SIZE, 
            hidden_size=HIDDEN_SIZE,
            device=DEVICE
        )
        print(f"✅ FreeGAN Generator loaded successfully")
        
        # Generate synthetic features
        noise_tensor = torch.randn(1, Z_DIM).to(DEVICE)
        
        with torch.no_grad():
            generated_features = generator(noise_tensor)
            gen_features = generated_features.cpu().numpy().flatten()
        
        print(f"\n🎯 GENERATION RESULTS:")
        print(f"   Generated features: {gen_features}")
        
        # Create a DataFrame for better visualization
        feature_names = list(INPUT_FEATURES.keys())
        result_df = pd.DataFrame([gen_features], columns=feature_names)
        print(f"\n📊 Generated Feature Values:")
        print(result_df.round(4))
        
        # Compare with your input features
        input_df = pd.DataFrame([list(INPUT_FEATURES.values())], columns=feature_names)
        print(f"\n📊 Your Input Features (for comparison):")
        print(input_df.round(4))
    
    else:
        print(f"❌ Unknown model type: {MODEL_TYPE}")
        
except FileNotFoundError as e:
    print(f"❌ Model file not found: {e}")
    print("💡 Make sure you have trained models first by running the training cells above")
    print("💡 Check that the model paths in MODEL_PATHS are correct")
    
except Exception as e:
    print(f"❌ Error during inference: {e}")
    print("💡 Check your model paths and input dimensions")
    
print("\n" + "=" * 60)
print("✨ Inference complete!")

📂 Loading model from: experiment/pointgan-Spectral_20241126_clean-20251008_010709/model/MLP_pipeline_1_p9_s0

🧠 MLP Model Inference
✅ Model loaded successfully
   - Input size: 10
   - Hidden size: 18
   - Output size: 2

📊 Input shape: torch.Size([1, 10])
📊 Input values: [0.5, 0.3, 0.8, 0.2, 0.9, 0.1, 0.6, 0.4, 0.7, 0.35]

🎯 PREDICTION RESULTS:
   Predicted X coordinate: 0.0230
   Predicted Y coordinate: -0.1661
   Predicted location: (0.0230, -0.1661)

✨ Inference complete!
